In [6]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.algorithms.gradients import *
from qiskit.algorithms.optimizers import *
from qiskit.circuit import Parameter
from qiskit.circuit.library import RealAmplitudes, ZZFeatureMap, EfficientSU2
from qiskit_machine_learning.algorithms.regressors import NeuralNetworkRegressor, VQR
from qiskit_machine_learning.neural_networks import EstimatorQNN, SamplerQNN
from qiskit.opflow import PauliSumOp, Z, StateFn, PauliOp, Zero, One
from qiskit.primitives import Estimator,Sampler
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPRegressor
from qiskit.quantum_info import SparsePauliOp, ScalarOp, Pauli
import matplotlib.pyplot as plt

In [7]:
n_samples = 300
n_features = 3
n_targets = 1

X, Y = make_regression(n_samples=n_samples, n_features=n_features, n_targets=n_targets, random_state=42)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)
Y_train = scaler.fit_transform(Y_train.reshape(-1,1)).flatten()
Y_test = scaler.fit_transform(Y_test.reshape(-1,1)).flatten()

In [8]:
param_x1 = Parameter("x1")
param_x2 = Parameter("x2")
param_x3 = Parameter("x3")

feature_map = QuantumCircuit(3)
feature_map.ry(param_x1, 0)
feature_map.ry(param_x2, 1)
feature_map.ry(param_x3, 2)
ansatz = RealAmplitudes(3, reps=3)

qc = QuantumCircuit(3)
qc.compose(feature_map, inplace=True)
qc.compose(ansatz, inplace=True)

In [9]:
qnn = EstimatorQNN(
    circuit=qc,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
)

regressor = NeuralNetworkRegressor(
    neural_network=qnn,
    loss="squared_error",
    optimizer=SLSQP(maxiter=100),
)
regressor.fit(X_train, Y_train)
regressor.score(X_test, Y_test)

0.927904780066153